# Лабораторная работа 4

Цель: исследовать связь между оптимизатором и качеством MLP-классификатора, проверить расписания learning rate, L2-регуляризацию и внешний baseline.

In [ ]:
from __future__ import annotations

from pathlib import Path

import optlib

ROOT = Path.cwd()
DATASETS = {
    "d1": ROOT / "data" / "first_dataset.csv",
    "d2": ROOT / "data" / "second_dataset.csv",
}

## Оптимизаторы и расписания

Сравниваем все методы первого порядка из лабораторий 1-2. Для MLP используется тот же C++ backprop и тот же плоский вектор параметров.

In [ ]:
def show(rows):
    try:
        import pandas as pd
    except ModuleNotFoundError:
        return rows
    return pd.DataFrame(rows)


rows = []
for name, path in DATASETS.items():
    for schedule in ["constant", "cosine"]:
        part = optlib.compare_nn_optimizers(
            path,
            methods=["gradient_descent", "heavy_ball", "nesterov", "adam", "rmsprop", "adagrad"],
            hidden_dim=12,
            learning_rate=0.03,
            max_iter=3000,
            schedule=schedule,
            seed=42,
        )
        for row in part:
            row["dataset"] = name
        rows.extend(part)
show(rows)

## Регуляризация

L2-регуляризация проверяется как простой ablation. Для закрытого d3 выбирается конфигурация, которая стабильно держит F1 на d1/d2 без переобучения.

In [ ]:
ablation_rows = []
for name, path in DATASETS.items():
    part = optlib.regularization_ablation(path, l2_values=[0.0, 1e-4, 1e-3], max_iter=3000)
    for row in part:
        row["dataset"] = name
    ablation_rows.extend(part)
show(ablation_rows)

## Внешний baseline

`sklearn.MLPClassifier` запускается только если установлен extra `experiments`. Он нужен как sanity-check, а не как зависимость C++ ядра.

In [ ]:
baselines = []
for name, path in DATASETS.items():
    baseline = optlib.sklearn_mlp_baseline(path, hidden_dim=12, max_iter=1000)
    if baseline is not None:
        baseline["dataset"] = name
        baselines.append(baseline)
show(baselines)

## d3 на защите

Закрытый набор скачивается через `download(file_id, dest)`. Затем используется тот же `evaluate(model, path, standardizer)`, где standardizer был обучен только на train-части d1/d2.

In [ ]:
model, split, metrics = optlib.train_binary_classifier(DATASETS["d1"], method="adam", max_iter=3000)
metrics
# optlib.download("<google-drive-id>", ROOT / "data" / "third_dataset.csv")
# optlib.evaluate(model, ROOT / "data" / "third_dataset.csv", split.standardizer)